# GPU-Accelerated Coin Counting — PUSL2101 Coursework

**Task:** Object counting (coins) from images.

**Pipeline in this notebook:**
1. Preprocessing
2. Classical computer-vision detector (feature extraction + Hough Circle Transform) — a fast, training-free baseline
3. Auto-labelling: turn the classical detector's output into YOLO-format bounding-box labels (since the raw dataset has no annotations)
4. Data augmentation to grow the small (<100 image) dataset
5. GPU-accelerated deep-learning detector: fine-tune YOLOv8n (Ultralytics/PyTorch, CUDA) on the augmented, auto-labelled set
6. Evaluation: classical vs. deep-learning counts vs. your own manual ground truth

**Running locally in VS Code:** see `README.md` for environment setup (venv + `pip install -r requirements.txt`).
If you have an NVIDIA GPU with CUDA drivers installed, training will use it automatically; otherwise it falls back to CPU (slower, but the notebook still runs end to end — just cut `epochs` down in Section 7 if training on CPU).


In [ ]:
import cv2
import numpy as np
import os, glob, shutil, random, json
import matplotlib.pyplot as plt
import torch

print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected - training will run on CPU (Section 7 will be slow).")

random.seed(42)
np.random.seed(42)


## 1. Load the dataset

Put all your raw coin images in one local folder and point `DATA_DIR` at it.


In [ ]:
# EDIT THIS to the local folder containing your coin images
DATA_DIR = r"./data/coin_images"

image_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpg")) +
                      glob.glob(os.path.join(DATA_DIR, "*.jpeg")) +
                      glob.glob(os.path.join(DATA_DIR, "*.png")))
print(f"Found {len(image_paths)} images in {DATA_DIR}")
assert len(image_paths) > 0, "No images found - check DATA_DIR"

# quick look at a few
fig, axes = plt.subplots(1, min(4, len(image_paths)), figsize=(16, 4))
for ax, p in zip(np.atleast_1d(axes), image_paths[:4]):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(os.path.basename(p)); ax.axis('off')
plt.tight_layout(); plt.show()
